# 03 — Exploratory Data Analysis (Phase 2)

Scope:

- Descriptive statistics and visualisations on **train + validation only**.
- Outliers, associations, and patterns.
- **Linear DR (PCA)** and **non-linear DR (UMAP)** on predictors only.
- Initial insights (model-development EDA is kept separate from the held-out test set).

Inputs:

- `data/processed/accepted_regression.parquet`
- `data/processed/accepted_classification.parquet`
- `outputs/split_definition.json` (chronological cutoffs)

Outputs: figures in `outputs/figures/` and tables in `outputs/tables/`.

### Note on target handling

The regression target `int_rate` is **included** in the descriptive-stats,
histogram, correlation-heatmap, and outlier tables (to describe the target and
rank predictors by their association with it). It is **excluded** from PCA and
UMAP, because dimensionality reduction is applied to the *predictor space*
only. Including the target would mix its variance into the components and
blur the loadings. This is a methodological distinction, not a leakage rule:
no test data is ever used.

In [1]:
import sys, json, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.eda_utils import LendingClubEDA
eda = LendingClubEDA()
from src.splits import ChronologicalSplitter

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
sns.set_theme(style='whitegrid')
OUT_FIG = PROJECT_ROOT / 'outputs' / 'figures'; OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_TABLES = PROJECT_ROOT / 'outputs' / 'tables'; OUT_TABLES.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT

PosixPath('/Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados')

## 1. Load cleaned data and reuse the frozen split

No re-splitting happens here. We only reuse the cutoffs recorded in Notebook 02.

In [2]:
with open(PROJECT_ROOT / 'outputs' / 'split_definition.json') as f:
    split_cfg = json.load(f)
split_cfg

{'date_col': 'issue_d', 'train_end': '2014-12', 'val_end': '2016-12'}

In [3]:
acc_reg = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'accepted_regression.parquet')
acc_clf = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'accepted_classification.parquet')
print('acc_reg:', acc_reg.shape, '| acc_clf:', acc_clf.shape)

acc_reg: (2260668, 152) | acc_clf: (1379602, 152)


In [4]:
splitter = ChronologicalSplitter(**split_cfg)

splits_reg = splitter.split(acc_reg)
splits_clf = splitter.split(acc_clf)

# EDA is done on train + validation ONLY. Test is untouched.
eda_reg = pd.concat([splits_reg.train, splits_reg.val], ignore_index=True)
eda_clf = pd.concat([splits_clf.train, splits_clf.val], ignore_index=True)
print('EDA regression rows:', len(eda_reg))
print('EDA classification rows:', len(eda_clf))

EDA regression rows: 1321847
EDA classification rows: 1129813


## 2. Define the EDA column sets

**Regression cohort:** we split the candidate columns into:

- `EDA_PREDICTORS_REG` — the predictor-only set used for PCA and UMAP.
- `EDA_TARGET_REG` — the regression target (`int_rate`), used only for
  descriptive and association tables.
- `EDA_DESCRIBE_REG` — predictors + target, used for descriptive statistics,
  histograms, correlation heatmaps, and IQR outlier counts.

**Classification cohort:** `EDA_NUMERIC` contains predictors only. The target
(`target_class`) is used as a hue for the UMAP plot, never as an input.

In [5]:
# ---- Predictor list (shared base) ------------------------------------------
EDA_NUMERIC = [
    'loan_amnt', 'term', 'annual_inc', 'dti', 'fico_range_low', 'fico_range_high',
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'total_acc', 'pub_rec',
    'pub_rec_bankruptcies', 'mort_acc', 'revol_bal', 'revol_util',
    'tot_cur_bal', 'tot_hi_cred_lim', 'bc_util', 'bc_open_to_buy',
    'mo_sin_old_rev_tl_op', 'num_actv_rev_tl', 'pct_tl_nvr_dlq',
]
EDA_NUMERIC = [c for c in EDA_NUMERIC if c in eda_reg.columns]

# ---- Regression-specific splits --------------------------------------------
# Predictors only — used for PCA and UMAP (no target involved)
EDA_PREDICTORS_REG = EDA_NUMERIC

# Target only — used for descriptive stats and correlation tables
EDA_TARGET_REG = ['int_rate'] if 'int_rate' in eda_reg.columns else []

# Combined view — used for describing the data, including the target
EDA_DESCRIBE_REG = EDA_PREDICTORS_REG + EDA_TARGET_REG

print('EDA_NUMERIC (predictors, classification cohort):', len(EDA_NUMERIC))
print('EDA_PREDICTORS_REG (for PCA/UMAP):', len(EDA_PREDICTORS_REG))
print('EDA_TARGET_REG:', EDA_TARGET_REG)
print('EDA_DESCRIBE_REG (predictors + target):', len(EDA_DESCRIBE_REG))

EDA_NUMERIC (predictors, classification cohort): 22
EDA_PREDICTORS_REG (for PCA/UMAP): 22
EDA_TARGET_REG: ['int_rate']
EDA_DESCRIBE_REG (predictors + target): 23


## 3. Descriptive statistics

Extended describe (count, missing, mean, std, quartiles, skew, kurtosis).
For regression, the table includes `int_rate` so we can characterise the
target. For classification, the target is categorical and described separately.

Saved to CSV for the mid-journey report.

In [6]:
desc_reg = eda.describe_numeric(eda_reg, EDA_DESCRIBE_REG)
desc_clf = eda.describe_numeric(eda_clf, EDA_NUMERIC)
desc_reg.to_csv(OUT_TABLES / 'eda_describe_regression.csv')
desc_clf.to_csv(OUT_TABLES / 'eda_describe_classification.csv')
desc_reg.head(25)

,count,mean,std,min,25%,50%,75%,max,missing_pct,skew,kurtosis
loan_amnt,1321847.0,14748.2240,8622.1427,500.00,8000.00,12900.00,20000.00,40000.00,0.0000,0.7092,-0.2190
term,1321847.0,42.8483,10.8379,36.00,36.00,36.00,60.00,60.00,0.0000,0.9507,-1.0962
annual_inc,1321843.0,76495.9420,69111.9196,0.00,46000.00,65000.00,90000.00,9573072.00,0.0003,44.1991,4426.1994
dti,1321773.0,18.3687,8.7961,0.00,12.04,17.83,24.24,822.00,0.0056,4.1527,261.5898
fico_range_low,1321847.0,695.0868,30.7992,610.00,670.00,690.00,710.00,845.00,0.0000,1.3067,1.8419
fico_range_high,1321847.0,699.0869,30.7998,614.00,674.00,694.00,714.00,850.00,0.0000,1.3069,1.8433
delinq_2yrs,1321818.0,0.3296,0.8920,0.00,0.00,0.00,0.00,39.00,0.0022,5.4097,53.9877
inq_last_6mths,1321817.0,0.6509,0.9577,0.00,0.00,0.00,1.00,33.00,0.0023,2.0674,10.4556
open_acc,1321818.0,11.6559,5.4694,0.00,8.00,11.00,14.00,97.00,0.0022,1.2995,3.3742
total_acc,1321818.0,25.0222,11.9035,1.00,16.00,23.00,32.00,176.00,0.0022,0.9460,1.6317


## 4. Histograms

Heavy-tailed columns (`annual_inc`, `revol_bal`, `tot_cur_bal`) are plotted on log1p.
The regression histogram grid includes `int_rate` so the target distribution is
visible alongside the predictors.

In [7]:
eda.plot_histograms(
    eda_reg, EDA_DESCRIBE_REG,
    out_path=OUT_FIG / 'histograms_regression.png',
    bins=50,
    log_scale_cols=['annual_inc', 'revol_bal', 'tot_cur_bal', 'tot_hi_cred_lim'],
)

INFO | Saved histogram grid -> /Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/outputs/figures/histograms_regression.png


In [8]:
eda.plot_histograms(
    eda_clf, EDA_NUMERIC,
    out_path=OUT_FIG / 'histograms_classification.png',
    bins=50,
    log_scale_cols=['annual_inc', 'revol_bal', 'tot_cur_bal', 'tot_hi_cred_lim'],
)

INFO | Saved histogram grid -> /Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/outputs/figures/histograms_classification.png


## 5. Class balance for the 3-class classification target

0 = Good, 1 = At risk, 2 = Default.

In [9]:
if 'target_class' in eda_clf.columns:
    counts = eda_clf['target_class'].value_counts().sort_index()
    pct = eda_clf['target_class'].value_counts(normalize=True).sort_index() * 100
    balance = pd.DataFrame({'count': counts, 'pct': pct.round(3)})
    balance.to_csv(OUT_TABLES / 'eda_class_balance.csv')
    print(balance)

    fig, ax = plt.subplots(figsize=(6, 4))
    balance['count'].plot(kind='bar', ax=ax, color=['#4C72B0','#DD8452','#C44E52'])
    ax.set_xticklabels(['0 Good', '1 At risk', '2 Default'], rotation=0)
    ax.set_title('Class balance (train+val)')
    fig.tight_layout()
    fig.savefig(OUT_FIG / 'class_balance.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

               count     pct
target_class                
0             899155  79.584
1              10102   0.894
2             220556  19.521


## 6. Relationship plots

- FICO vs `int_rate` (regression target).
- DTI vs `int_rate`.

Here `int_rate` is used as the y-axis, which is fine because we are plotting
the target against a predictor — no information is fed into a model.

In [10]:
if 'fico_range_low' in eda_reg.columns and 'int_rate' in eda_reg.columns:
    eda.plot_scatter(eda_reg, x='fico_range_low', y='int_rate',
                 out_path=OUT_FIG / 'scatter_fico_vs_int_rate.png',
                 sample_n=20000)

INFO | Saved scatter -> /Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/outputs/figures/scatter_fico_vs_int_rate.png


In [11]:
if 'dti' in eda_reg.columns and 'int_rate' in eda_reg.columns:
    eda.plot_scatter(eda_reg, x='dti', y='int_rate',
                 out_path=OUT_FIG / 'scatter_dti_vs_int_rate.png',
                 sample_n=20000)

INFO | Saved scatter -> /Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/outputs/figures/scatter_dti_vs_int_rate.png


## 7. Correlation heatmap

Spearman correlation (robust to non-linearity and outliers). The regression
heatmap includes `int_rate` so predictors can be ranked by association with
the target. This is a **descriptive** step and does not inform PCA or any
model directly.

In [12]:
corr = eda.plot_correlation_heatmap(
    eda_reg, EDA_DESCRIBE_REG,
    out_path=OUT_FIG / 'corr_heatmap_regression.png',
    method='spearman',
)
corr.to_csv(OUT_TABLES / 'corr_spearman_regression.csv')
corr.round(2).head(10)

INFO | Saved heatmap -> /Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/outputs/figures/corr_heatmap_regression.png


,loan_amnt,term,annual_inc,dti,fico_range_low,fico_range_high,delinq_2yrs,inq_last_6mths,open_acc,total_acc,...,revol_bal,revol_util,tot_cur_bal,tot_hi_cred_lim,bc_util,bc_open_to_buy,mo_sin_old_rev_tl_op,num_actv_rev_tl,pct_tl_nvr_dlq,int_rate
loan_amnt,1.00,0.43,0.50,0.05,0.13,0.13,0.01,-0.03,0.21,0.24,...,0.49,0.12,0.34,0.37,0.05,0.23,0.20,0.17,0.07,0.12
term,0.43,1.00,0.14,0.09,0.02,0.02,-0.00,-0.00,0.09,0.11,...,0.17,0.08,0.16,0.16,0.07,0.02,0.06,0.07,0.03,0.40
annual_inc,0.50,0.14,1.00,-0.21,0.11,0.11,0.09,0.03,0.25,0.32,...,0.41,0.07,0.53,0.56,0.01,0.22,0.25,0.13,-0.06,-0.13
dti,0.05,0.09,-0.21,1.00,-0.03,-0.03,-0.01,-0.00,0.32,0.24,...,0.25,0.17,0.12,0.12,0.17,-0.06,0.04,0.26,0.08,0.18
fico_range_low,0.13,0.02,0.11,-0.03,1.00,1.00,-0.21,-0.10,0.04,0.03,...,0.08,-0.36,0.17,0.24,-0.38,0.46,0.12,-0.15,0.35,-0.41
fico_range_high,0.13,0.02,0.11,-0.03,1.00,1.00,-0.21,-0.10,0.04,0.03,...,0.08,-0.36,0.17,0.24,-0.38,0.46,0.12,-0.15,0.35,-0.41
delinq_2yrs,0.01,-0.00,0.09,-0.01,-0.21,-0.21,1.00,0.02,0.06,0.13,...,-0.06,-0.02,0.09,0.09,-0.02,-0.05,0.11,-0.00,-0.49,0.05
inq_last_6mths,-0.03,-0.00,0.03,-0.00,-0.10,-0.10,0.02,1.00,0.13,0.14,...,-0.05,-0.08,0.03,0.03,-0.07,0.02,-0.01,0.11,-0.04,0.22
open_acc,0.21,0.09,0.25,0.32,0.04,0.04,0.06,0.13,1.00,0.70,...,0.38,-0.15,0.35,0.39,-0.12,0.30,0.15,0.65,0.05,-0.01
total_acc,0.24,0.11,0.32,0.24,0.03,0.03,0.13,0.14,0.70,1.00,...,0.31,-0.11,0.41,0.44,-0.09,0.23,0.34,0.38,-0.08,-0.05


## 8. Linear dimensionality reduction: PCA

**Fit on predictors only** (`EDA_PREDICTORS_REG`). The target `int_rate` is
excluded so that PC1, PC2, etc. describe the structure of the predictor space,
which is what Phase 4 will feed into the models.

Outputs:
- Scree plot
- 2D projection (PC1 vs PC2)
- Loadings table (rows = predictors, columns = PCs)

The scaler and dimensionality reducer are fit only on a complete-case training
sample. Validation rows are projected using transform(), without refitting.
Complete-case filtering may underrepresent early vintages with missing bureau
fields; these plots describe the retained training sample, not the entire population.

In [13]:
pca_info = eda.run_pca(
    splits_reg.train, EDA_PREDICTORS_REG,      # predictors only — target excluded
    out_dir=OUT_FIG / 'pca',
    n_components=10,
    sample_n=50000,
    seed=42,
)

# Transform a reproducible validation sample; never refit the scaler or PCA.
pca_validation_input = eda.reproducible_sample(splits_reg.val[EDA_PREDICTORS_REG].dropna(), n=50000)
pca_validation = eda.transform_pca(pca_validation_input)
pca_validation.to_parquet(PROJECT_ROOT / 'data' / 'processed' / 'pca_validation_sample.parquet')
assert pca_validation.index.equals(pca_validation_input.index)
print('PCA fit: training only; validation projected rows:', len(pca_validation))

PCA fit: training only; validation projected rows: 50000


In [14]:
evr = pd.Series(
    pca_info['explained_variance_ratio'],
    index=[f'PC{i+1}' for i in range(len(pca_info['explained_variance_ratio']))],
    name='explained_variance_ratio',
)
cum = pd.Series(
    pca_info['cumulative'],
    index=evr.index, name='cumulative',
)
pca_table = pd.concat([evr, cum], axis=1).round(4)
pca_table.to_csv(OUT_TABLES / 'pca_explained_variance.csv')
pca_table

,explained_variance_ratio,cumulative
PC1,0.1963,0.1963
PC2,0.1448,0.3411
PC3,0.0944,0.4354
PC4,0.0885,0.5240
PC5,0.0705,0.5945
PC6,0.0526,0.6471
PC7,0.0494,0.6964
PC8,0.0457,0.7422
PC9,0.0391,0.7813
PC10,0.0354,0.8167


### 8.1 PCA loadings

Rows are predictors, columns are components. Since the target is excluded,
each loading can be interpreted as "how much this predictor contributes to
this component". A predictor with high loadings across the first two
components dominates the projection.

In [15]:
loadings = pd.read_csv(OUT_FIG / 'pca' / 'pca_loadings.csv', index_col=0)
loadings.round(3).head(25)

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
loan_amnt,0.256,0.146,-0.146,0.033,-0.105,0.455,0.224,-0.147,0.006,0.020
term,0.098,0.120,-0.076,0.057,-0.110,0.669,0.347,0.235,-0.156,-0.062
annual_inc,0.262,0.120,-0.116,-0.225,-0.043,-0.062,0.199,-0.393,0.046,-0.014
dti,0.018,0.148,0.117,0.391,0.006,0.063,-0.238,0.456,-0.078,0.475
fico_range_low,0.262,-0.386,-0.159,0.083,-0.059,0.048,-0.146,0.137,0.055,0.058
fico_range_high,0.262,-0.386,-0.159,0.083,-0.059,0.048,-0.146,0.137,0.055,0.058
delinq_2yrs,0.001,0.109,0.123,-0.250,0.507,0.174,-0.089,0.017,-0.179,0.295
inq_last_6mths,0.035,0.026,0.184,-0.068,0.033,-0.195,0.577,0.297,0.646,0.247
open_acc,0.245,0.126,0.390,0.302,0.081,-0.094,0.068,-0.057,-0.166,-0.111
total_acc,0.267,0.153,0.355,0.081,0.091,-0.016,-0.046,0.169,-0.011,-0.301


### 8.2 Check — the loadings table should contain no `int_rate` row.

In [16]:
assert 'int_rate' not in loadings.index, 'int_rate must not appear in PCA loadings'
print('OK: int_rate is not part of the PCA. Loadings rows:', len(loadings.index))

OK: int_rate is not part of the PCA. Loadings rows: 22


## 9. Non-linear dimensionality reduction: UMAP

Fit on predictors only. The classification target is used **only as a hue**
for coloring — it is not an input to UMAP. This lets us visually inspect
whether the classes separate in the predictor space without contaminating
the embedding.

The scaler and dimensionality reducer are fit only on a complete-case training
sample. Validation rows are projected using transform(), without refitting.
Complete-case filtering may underrepresent early vintages with missing bureau
fields; these plots describe the retained training sample, not the entire population.

In [17]:
try:
    emb = eda.run_umap(
        splits_clf.train, EDA_NUMERIC,           # predictors only
        out_dir=OUT_FIG / 'umap',
        hue='target_class',             # target used only for coloring
        n_neighbors=30,
        min_dist=0.1,
        sample_n=30000,
        seed=42,
    )
    emb.to_parquet(PROJECT_ROOT / 'data' / 'processed' / 'umap_classification_sample.parquet')
    validation_input = eda.reproducible_sample(splits_clf.val[EDA_NUMERIC].dropna(), n=30000)
    validation_emb = eda.transform_umap(validation_input)
    validation_emb['target_class'] = splits_clf.val.loc[validation_emb.index, 'target_class']
    validation_emb.to_parquet(PROJECT_ROOT / 'data' / 'processed' / 'umap_validation_sample.parquet')
    print('UMAP fit: training only; validation projected rows:', len(validation_emb))
    emb.head()
except ImportError as e:
    print('UMAP not available:', e)
    print('Install with: pip install umap-learn')

/Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP fit: training only; validation projected rows: 30000


## 10. Outlier inspection

Quick IQR-based outlier counts per numeric column (documentation only — we do
not clip here; clipping is a learned transform and must be fit on training folds).

For regression, the table includes `int_rate` so we can note how many outlier
target values exist. This is descriptive and does not affect any model.

In [18]:

outliers = pd.Series({c: eda.iqr_outliers(eda_reg[c]) for c in EDA_DESCRIBE_REG})
outliers = outliers.sort_values(ascending=False)
outliers.to_csv(OUT_TABLES / 'eda_iqr_outliers_regression.csv')
outliers.head(15)

delinq_2yrs             262841
pub_rec                 218702
pub_rec_bankruptcies    153809
bc_open_to_buy          110641
revol_bal                77152
pct_tl_nvr_dlq           75848
inq_last_6mths           70905
annual_inc               65035
open_acc                 45684
tot_cur_bal              43163
tot_hi_cred_lim          42622
fico_range_low           39277
fico_range_high          39277
num_actv_rev_tl          37822
mo_sin_old_rev_tl_op     34974
dtype: int64

## 11. Data-quality flags

Negative DTI and the fixed 999 sentinel were mapped to NaN during parsing.
Zero income is retained, but is mapped to NaN as a ratio denominator.
Utilization over 100% is flagged, not automatically discarded: over-limit balances
can be real. No data-derived clipping thresholds are estimated on validation/test.
Heavy-tailed nonnegative variables, including zeros, use log1p for plotting only.

In [19]:
flags = []
if 'dti' in eda_reg.columns:
    flags.append({'flag': 'dti == -1 (sentinel)',
                  'count': int((eda_reg['dti'] == -1).sum())})
    flags.append({'flag': 'dti == 999 (sentinel)',
                  'count': int((eda_reg['dti'] == 999).sum())})
if 'annual_inc' in eda_reg.columns:
    flags.append({'flag': 'annual_inc == 0',
                  'count': int((eda_reg['annual_inc'] == 0).sum())})
if 'revol_util' in eda_reg.columns:
    flags.append({'flag': 'revol_util > 100',
                  'count': int((eda_reg['revol_util'] > 100).sum())})
if 'int_rate' in eda_reg.columns:
    flags.append({'flag': 'int_rate == max value',
                  'count': int((eda_reg['int_rate'] == eda_reg['int_rate'].max()).sum())})

flags_df = pd.DataFrame(flags)
flags_df.to_csv(OUT_TABLES / 'eda_data_quality_flags.csv', index=False)
flags_df

,flag,count
0,dti == -1 (sentinel),0
1,dti == 999 (sentinel),0
2,annual_inc == 0,59
3,revol_util > 100,5090
4,int_rate == max value,133


## 12. Initial insights

Income and balances are strongly right-skewed, so log1p plots reveal their central
ranges more clearly than raw-scale histograms. FICO is negatively associated with
interest rate and term is positively associated; the following cell reports the
current correlations and training-only PCA variance directly from this run.
UMAP islands contain overlapping outcomes rather than clean repayment classes.
Class imbalance and differences in product/year composition limit predictive and
causal interpretation. Complete-case DR sampling is an additional selection limit.
Neither PCA variance nor visual islands establish useful supervised performance.

## 13. Development boundaries

Descriptive plots may summarize training plus validation. All fitted scaling,
PCA and UMAP use training only, and validation projections reuse those fitted
objects. The held-out test set contributes to none of these outputs.

In [20]:
print('FICO / interest-rate Spearman:', corr.loc['fico_range_low', 'int_rate'])
print('Term / interest-rate Spearman:', corr.loc['term', 'int_rate'])
print('Training PCA: PC1 variance:', float(pca_info['explained_variance_ratio'][0]))
print('Training PCA: cumulative variance at 10 PCs:', float(pca_info['cumulative'][-1]))
print('Training complete cases for PCA:', len(splits_reg.train[EDA_PREDICTORS_REG].dropna()), '/', len(splits_reg.train))


FICO / interest-rate Spearman: -0.4056085121118269
Term / interest-rate Spearman: 0.39794498840350845
Training PCA: PC1 variance: 0.19625054140078807
Training PCA: cumulative variance at 10 PCs: 0.8167243620856488
Training complete cases for PCA: 391972 / 466345
